In [1]:
# LIBRARY and GLOBAL IMPORTS
import os, ray, sys, re, glob, csv
sys.path.append(".")
from functionsG import *

# === Core Libraries ===
# Import necessary libraries
import numpy as np
from collections import Counter
from collections import defaultdict, deque
import pandas as pd

# === Directories ===
var_dir = './running_vars'
uic_dir = './big_dfs/2_UIC_hier_data'

def create_df_suff_lkup(df_lkup):
    df_lkup_unique_suff = df_lkup.copy()
    df_lkup_unique_suff['uic_suff'] = df_lkup_unique_suff['asg_uic'].str[-2:]
    df_lkup_unique_suff['uic_pde_suff'] = df_lkup_unique_suff['asg_uic_pde'].str[-4:]
    df_lkup_unique_suff = df_lkup_unique_suff.drop(columns=['asg_uic_pde','asg_uic']).drop_duplicates()
    suff_dict = df_lkup_unique_suff.uic_suff.value_counts().to_dict()
    unique_suffs = [suff for suff in suff_dict if suff_dict[suff] == 1]
    df_suff_lkup = df_lkup_unique_suff[df_lkup_unique_suff.uic_suff.isin(unique_suffs)]
    return df_suff_lkup

df_suff_lkup = create_df_suff_lkup(load_feather('df_lu_uic_1to1'))
store_feather(df_suff_lkup,'df_suff_lkup',uic_dir)

# UIC Hierarchy Analysis Functions
def find_subordinate_uics_recursive(df_in, 
                                   top_uic, 
                                   uic_col = 'UIC', 
                                   parent_col = 'PARENTUIC',
                                   include_top = True,
                                   max_depth = None,
                                   show=False):
    """
    Recursively find all subordinate UICs for a given top-level UIC.
    
    Parameters:
    -----------
    df : pd.DataFrame
    DataFrame containing UIC hierarchy data
    top_uic : str, top-level UIC to get all subordinates for
    List of top-level UICs to start the search from
    uic_col : str, default 'UIC'
    Column name containing the unit identification codes
    parent_col : str, default 'PARENTUIC'
    Column name containing the parent/superior UIC for each unit
    include_top : bool, default True
    Whether to include the top-level UICs in the results
    max_depth : Optional[int], default None
    Maximum depth to search (None for unlimited depth)
    
    Returns:
    --------
    sub_uics : list
    list of all subordinate UICs
    """
    df=df_in.copy()
    # Validate inputs
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if uic_col not in df.columns:
        raise ValueError(f"Column '{uic_col}' not found in DataFrame")
    if parent_col not in df.columns:
        raise ValueError(f"Column '{parent_col}' not found in DataFrame")
    
    # Remove rows with null UICs or parent UICs for cleaner processing
    clean_df = df.dropna(subset=[uic_col, parent_col])
    
    # Create lookup dictionary for faster searching
    # Group by parent UIC to get all direct subordinates
    hierarchy_dict = defaultdict(set)
    
    # Build the hierarchy dictionary
    for _, row in clean_df.iterrows():
        parent = row[parent_col]
        child = row[uic_col]
        if pd.notna(parent) and pd.notna(child):
            hierarchy_dict[parent].add(child)
    
    if show:
        print(f"📊 Built hierarchy dictionary with {len(hierarchy_dict)} parent UICs")

    subordinates = set()
    
    if include_top:
        subordinates.add(top_uic)
    
    # Use BFS to find all subordinates
    queue = deque([(top_uic, 0)])  # (uic, depth)
    visited = set()
    
    while queue:
        # print('fy, queue',clean_df.FY.unique().tolist(),queue)
        current_uic, depth = queue.popleft()
        
        # Skip if we've already processed this UIC
        if current_uic in visited:
            continue
            
        visited.add(current_uic)

        # Check depth limit
        if max_depth is not None and depth >= max_depth:
            continue
        
        # Get direct subordinates
        direct_subordinates = hierarchy_dict.get(current_uic, set())
        
        # Add them to df_uic_hierarchy.sub_class.unique()results and queue
        for sub_uic in direct_subordinates:
            subordinates.add(sub_uic)
            queue.append((sub_uic, depth + 1))
    
    if show:
        fy_var = None
        if 'FY' in clean_df.columns:
            fy_var ='FY'
        elif 'fy' in clean_df.columns:
            fy_var = 'fy'
        if fy_var:
            print(f"🎯 Found {len(subordinates)} total subordinates for {top_uic} for FY {clean_df[fy_var].unique().tolist()}")
        else:
            print("!!!!!!!!!!  No recognizable Fiscal Year Column found in DataFrame")
    return subordinates

 df_lu_uic_1to1 Loaded!!  - (00.15 seconds and 0.629 MB of memory)
 df_suff_lkup Stored!!  - (00.02 seconds and 0.019 MB of memory)


In [ ]:
build_fy_uic_dict = True
if build_fy_uic_dict:
    df_w = load_feather('./2_UIC_hier_data/df_WDARFF_2015-2026')
    fy_uic_dict = dict(); count = 0
    total_computes = df_w.fy.nunique() * len(uic_subordinate_dict_raw)
    
    for uic in list(uic_subordinate_dict_raw.keys()):
        fy_uic_dict[uic]={'top_name':uic_subordinate_dict_raw[uic]['top_name'],
                          'class':uic_subordinate_dict_raw[uic]['class'],
                          'sub_class':uic_subordinate_dict_raw[uic]['sub_class'],
                          'fy':dict()}
        for fyr in sorted(df_w.fy.unique().tolist()):
            fy_uic_dict[uic]['fy'][fyr] = []
            df_fy = df_w[df_w.fy == fyr].copy()
            sub_uics = find_subordinate_uics_recursive(df_fy,uic) 
            count += 1
            print(f"Computing {count} of {total_computes} - UIC ===>  {uic} ({uic_subordinate_dict_raw[uic]['top_name']}), For FY {fyr} has {len(sub_uics)} ")
            fy_uic_dict[uic]['fy'][fyr] = list(set(sub_uics))
    store_json(fy_uic_dict,'fy_uic_dict',var_dir)
else:
    fy_uic_dict = load_json('fy_uic_dict', var_dir)


In [ ]:
df_w = df_w[['RELDPTH', 'PARENTUIC', 'UIC', 'EDATE', 'SHORTDSCR', 'LONGDSCR',
       'ORGTYPENAME', 'AUTHTOT', 'CATCODE',
       'SERVICE', 'GFMCOMPO', 'UTCAT', 'UTARMYCAT', 'UTSIZE', 'UTGFMSIZE',
       'ID', 'PARENTID', 'FYDATE', 'fy']]
df_w.head()

In [6]:
new_uic_dict_to_df = {'asg_uic':[],'fy':[],'parent_uic':[],'top_name':[]}
new_uic_dict_to_df

{'asg_uic': [], 'fy': [], 'parent_uic': [], 'top_name': []}

In [ ]:
df_w[df_w.SHORTDSCR == 'SDDC']

In [8]:
# Now populate uic_subordinate_dict_raw with data from FMS Web using the get_file_fy and 
# find_subordinate_uics_recursive functions to create uic_subordinate_dict
def get_file_fy(filename):
    pattern = 'ORGS3' + r'(\d{2})'
    match = re.search(pattern,filename)
    return int('20'+ match.group(1))

# Load the uic Lookup dictionary to populate the uic_pde columns for use with our tables
df_uic_lookup = load_feather('df_uic_lookup')

# Initiate the final uis subordinate dictionary
uic_subordinate_dict = uic_subordinate_dict_raw.copy()

# Identify the FMS Web data directory structure and loop through it
# directory by directory,  file by file
uic_data_dir = './data_imported/hierarchy_data/UICs2'
sub_dirs = ['HDIV','LDIV','SFAB','RGR','SOAR','SFG']
file_dict = dict()

action="Looping through data directories to build uic_subordinate_dict"
data_loop = time_start(action, nest=2)
for sub_dir in sub_dirs:
    sub_action=f"working sub-directory {sub_dir}"
    sub_loop = time_start(sub_action,nest=4)
    sub_dir_files = os.listdir(os.path.join(uic_data_dir, sub_dir))
    sub_dir_files = [file_name for file_name in sub_dir_files if file_name.endswith('csv')]
    for file_name in sub_dir_files:
        dfh = pd.read_csv(os.path.join(uic_data_dir,sub_dir,file_name))
        fy = get_file_fy(file_name)
        # Populate the SOF entries in uic_subordinate_dict
        for top_uic in uic_subordinate_dict:
            if uic_subordinate_dict[top_uic]['class'] == sub_dir:
                try:
                    uic_subordinate_dict[top_uic]['sub_units_by_fy'].update({fy:find_subordinate_uics_recursive(dfh,top_uic,show=False)})
                    # print(f" Updated uic_subordinate_dict[{top_uic}] ({sub_dir}) for fy {fy}, with subordinate units")
                except Exception as e:
                    print(f"Failed to find subordinate units for {top_uic} fy{fy} file: {file_name}: {e}")
    time_stop(sub_loop,action=sub_action,nest=4)
time_stop(data_loop,action=action,nest=2)
store_pickle(uic_subordinate_dict,'uic_subordinate_dict',var_dir)
print("Storing 'uic_subordinate_dict'... ")

# Now transform the uic_subordinate_dict into a lookup dataframe called df_uic_hierarchy
df_uic_hierarchy= pd.DataFrame.from_dict(uic_subordinate_dict, orient='index').reset_index()
df_uic_hierarchy = df_uic_hierarchy.rename(columns={'index':'top_uic', 'uic_pde':'top_uic_pde', 'description':'top_uic_description'})
df_uic_hierarchy = (df_uic_hierarchy.assign(sub_units_by_fy=df_uic_hierarchy['sub_units_by_fy'].map(dict.items))
         .explode('sub_units_by_fy')
         .assign(fy=lambda x: x['sub_units_by_fy'].str[0],
                 asg_uic=lambda x: x['sub_units_by_fy'].str[1])
         .explode('asg_uic')
         .drop(columns='sub_units_by_fy')
         .reset_index(drop=True)
        )
df_uic_hierarchy = df_uic_hierarchy.merge(df_uic_lookup, how='left', on=['asg_uic'])
df_uic_hierarchy = df_uic_hierarchy[[df_uic_hierarchy.columns.tolist()[-1]]+df_uic_hierarchy.columns.tolist()[:-1]]
df_uic_hierarchy = move_column_after(df_uic_hierarchy,'asg_uic','asg_uic_pde')
df_uic_hierarchy = move_column_after(df_uic_hierarchy,'top_uic_pde','asg_uic')
df_uic_hierarchy = move_column_after(df_uic_hierarchy,'top_uic','top_uic_pde')
df_uic_hierarchy = df_uic_hierarchy.sort_values(by=['asg_uic_pde','fy' ])
# Store df_uic_hierarchy
store_feather(df_uic_hierarchy,'df_uic_hierarchy_old_school',uic_dir)

# Now build and save uic_lookup_dict
df_dict = df_uic_hierarchy.copy().dropna(subset='asg_uic_pde')
df_dict = df_dict[['asg_uic','asg_uic_pde']].dropna(subset=['asg_uic','asg_uic_pde']).drop_duplicates()
uic_lookup_dict = dict(zip(df_dict.asg_uic, df_dict.asg_uic_pde))
store_json(uic_lookup_dict,'uic_lookup_dict',var_dir)
print("Storing 'uic_lookup_dict'... ")

## Create df_hier_18_26 by concatenating all of the tables for each year from 2018-2026
hier_df_list = []
for fy in range(18,27):
    uic_data_dir = './data_imported/hierarchy_data/UICs2'
    sub_dir = 'DIV'
    file_name = f'LAU_PR_ORGS3{fy}DIV28.csv'
    dfh_old = pd.read_csv(os.path.join(uic_data_dir,sub_dir,file_name))
    hier_df_list.append(dfh_old)

df_hier_18_26 = pd.concat(hier_df_list)
store_feather(df_hier_18_26,'df_hier_18_26',uic_dir)

 df_uic_lookup Loaded!!  - (00.14 seconds and 0.762 MB of memory)
__________Start Looping through data directories to build uic_subordinate_dict at 12:01:02 (EST) Tue, 03 Feb 2026
____________________Start working sub-directory HDIV at 12:01:02 (EST) Tue, 03 Feb 2026
____________________...working sub-directory HDIV complete. duration: 08.56 seconds at 12:01:11 (EST) Tue, 03 Feb 2026
____________________Start working sub-directory LDIV at 12:01:11 (EST) Tue, 03 Feb 2026
____________________...working sub-directory LDIV complete. duration: 06.27 seconds at 12:01:17 (EST) Tue, 03 Feb 2026
____________________Start working sub-directory SFAB at 12:01:17 (EST) Tue, 03 Feb 2026
____________________...working sub-directory SFAB complete. duration: 38.22 seconds at 12:01:55 (EST) Tue, 03 Feb 2026
____________________Start working sub-directory RGR at 12:01:55 (EST) Tue, 03 Feb 2026
____________________...working sub-directory RGR complete. duration: 00.60 seconds at 12:01:56 (EST) Tue, 03 Feb

In [9]:
unit_level_dict = load_json('unit_level_dict',var_dir)
unit_to_division_name_dict = load_json('unit_to_division_name_dict',var_dir)
unit_to_division_uic_dict = load_json('unit_to_division_uic_dict',var_dir)
unit_to_division_uic_pde_dict = load_json('unit_to_division_uic_pde_dict',var_dir)
uic_div_tups_list = load_json('uic_div_tups_list',var_dir)
uic_div_tups_dict = {'uic_pde':[tup[0] for tup in uic_div_tups_list],
                     'uic':[tup[1] for tup in uic_div_tups_list],
                     'div_name':[tup[2] for tup in uic_div_tups_list]}
df_div_tups = pd.DataFrame(uic_div_tups_dict)

In [ ]:
# Create a uic_hierarchy_fy_list to hold the indiviual dataframes for each fiscal year
uic_hierarchy_fy_list = []
div_uics = [stop]
raw_uic_dict = {'div_uics':div_uics}
raw_sub_dict = dict() 
loop_tups = [('div_uics',1,('UICTYPE','TITULAR',True),'bde_uics','Divisions','Brigades'),
             ('bde_uics',1,('UICTYPE','PARENT',True),'bn_uics','Brigades','Battalions'),
             ('bn_uics',1,('ORG','DIV',True),'co_uics','Battalions','Companies')]
show=False; nest = 1;
fy_start = 18; fy_end = 26 ; fy_rng = (fy_start!=fy_end)

actionB = "Building the df_uic_hierarchy_master DataFrame for Fiscal Year{}"\
    .format(f's 20{fy_start}-20{fy_end}' if fy_rng else f' {fy}')
bigbigbigl = time_start(actionB,nest=nest*1)
actionG = "Giant Loop through Fiscal Year{}".format('s' if fy_rng else '')
bigbigl = time_start(actionG,nest = nest*2)
for fy in range(fy_start,fy_end+1):
    int_fy = int(f'20{fy}')
    action2 = f'Big Loop for Fiscal Year 20{fy}'
    bigl = time_start(action2,nest = nest*3)
    # Read in the fiscal year data from the older tables
    file_name = f'LAU_PR_ORGS3{fy}DIV28.csv'
    dfh_old = pd.read_csv(os.path.join(uic_data_dir,sub_dir,file_name))
    
    # Read in the fiscal year data from the newer tables
    uic_data_dir2 = './data_imported/hierarchy_data/HUICs'
    file_name2 = f'xxxx_20{fy}.txt'
    dfh_new = pd.read_csv(os.path.join(uic_data_dir2,file_name2),sep='\t')

    # Reduce the DataFrame for speed
    dfh_new = dfh_new[dfh_new.SERVICE == 'ARMY'].reset_index(drop=True)[['UIC','UTSIZE']].copy()
    
    # Perform a join to add the UTSIZE column to df_old
    dfh_old = dfh_old.merge(dfh_new,how='left',on='UIC')
    for tup in loop_tups:
        action3 = f"Loop through Fiscal Year 20{fy} {tup[4]} to find {tup[5]}..."
        msg ='_____'*1
        smlp = time_start(action3,nest=nest*4)
        col_tup = tup[2]
        raw_uic_dict[tup[3]]=[]
        for top_uic in raw_uic_dict[tup[0]]:
            if top_uic in dfh_old.UIC.unique().tolist():
                max_depth = tup[1]
                include_top = False
                show = False
                uic_col = 'UIC'
                parent_col = 'PARENTUIC'
                sub_list = find_subordinate_uics_recursive(dfh_old,top_uic, uic_col, parent_col, include_top, max_depth, show)
                raw_sub_dict[top_uic] = sub_list
                simp_name = dfh_old[dfh_old.UIC == top_uic]['DRRSANAME'].iloc[0]
                if show:
                    msg ='_____'*3
                    print(msg+f"For uic {top_uic}, ({simp_name}) we found {len(sub_list):,} subordinate units at a maximum depth of {max_depth}.")
                dfh_oldop = dfh_old[(dfh_old.UIC.isin(sub_list)) & (dfh_old[col_tup[0]] == col_tup[1])]
                raw_uic_dict[tup[3]] += dfh_oldop.UIC.unique().tolist()
            else:
                print(msg+f"=====> !!!!! UIC '{top_uic}' not found in the FMS Web Table for fiscal year 20{fy}")
        time_stop(smlp,action=action3,nest=nest*4)
    

    ### Build the concatenated DataFrame
    
    # Create a DataFrame from the bde_uics and call it df_bde_cmd
    df_bde_cmd = dfh_old[dfh_old.UIC.isin(raw_uic_dict['bde_uics'])].copy()
    # Assign the NODETYPE: BDE_CMD
    df_bde_cmd['NODETYPE'] = 'BDE_CMD'
    
    # Create a DataFrame from the bn_uics and call it df_bn_cmd
    df_bn_cmd = dfh_old[dfh_old.UIC.isin(raw_uic_dict['bn_uics'])].copy()
    # Assign the NODETYPE: BDE_CMD
    df_bn_cmd['NODETYPE'] = 'BN_CMD'
    # Create a DataFrame by filtering for direct reporting companies to BDE  and call it df_co_bde_dir
    df_co_bde_dir = df_bn_cmd[df_bn_cmd.SIMPLENAME.str.contains('Co')].copy()
    # Assign the NODETYPE: CO_BDE_DIR
    df_co_bde_dir['NODETYPE'] = 'CO_BDE_DIR'
    # Filter the df_co_bde_dir UICs out of df_bn_cmd
    df_bn_cmd = df_bn_cmd[~df_bn_cmd.UIC.isin(df_co_bde_dir.UIC.unique())].copy()
    # Create a DataFrame by filtering for BDE HHC's and call it df_hhc_bde
    df_hhc_bde = df_bn_cmd[df_bn_cmd.SIMPLENAME.str.contains('HH')].copy()
    # Assign the NODETYPE: HHC_BDE
    df_hhc_bde['NODETYPE'] = 'HHC_BDE'
    # Filter the df_hhc_bde UICs out of df_bn_cmd
    df_bn_cmd = df_bn_cmd[~df_bn_cmd.UIC.isin(df_hhc_bde.UIC.unique())].copy()
    # Create a DataFrame by filtering for BDE direct detachments and call it df_det_bde
    df_det_bde = df_bn_cmd[df_bn_cmd.SIMPLENAME.str.contains('De')].copy()
    # Assign the NODETYPE: DET_BDE_DIR
    df_det_bde['NODETYPE'] = 'DET_BDE_DIR'
    # Filter the df_det_bde UICs out of df_bn_cmd
    df_bn_cmd = df_bn_cmd[~df_bn_cmd.UIC.isin(df_det_bde.UIC.unique())].copy()
    # Create a DataFrame by filtering for STB HQ's and call it df_hqstb
    df_hqstb = df_bn_cmd[df_bn_cmd.SIMPLENAME.str.contains('HQSTB|HQ STB')].copy()
    # Assign the NODETYPE: HQSTB
    df_hqstb['NODETYPE'] = 'HQSTB'
    # Filter the df_hqstb UICs out of df_bn_cmd
    df_bn_cmd = df_bn_cmd[~df_bn_cmd.UIC.isin(df_hqstb.UIC.unique())].copy()
    # Create a DataFrame by filtering for BDE direct aviation companies and call it df_co_bde_dir_av
    df_co_bde_dir_av = df_bn_cmd[df_bn_cmd.DRRSANAME.str.contains('AV CO ',na=False)].copy()
    # Assign the NODETYPE: CO_BDE_DIR_AV
    df_co_bde_dir_av['NODETYPE'] = 'CO_BDE_DIR_AV'
    # Filter the df_co_bde_dir_av UICs out of df_bn_cmd
    df_bn_cmd = df_bn_cmd[~df_bn_cmd.UIC.isin(df_co_bde_dir_av.UIC.unique())].copy()
    
    # Create a DataFrame from the co_uics and call it df_co_cmd
    df_co_cmd = dfh_old[dfh_old.UIC.isin(raw_uic_dict['co_uics'])].copy()
    # Assign the NODETYPE: CO_CMD
    df_co_cmd['NODETYPE'] = 'CO_CMD'
    # Create a DataFrame by filtering for aviation FSC's it df_fsc_dir_av
    df_fsc_dir_av = df_co_cmd[df_co_cmd.SIMPLENAME == 'FSC'].copy()
    # NOTE: These answer directly to the manuever battalions so they're good!
    # Filter the df_fsc_dir_av UICs out of df_co_cmd
    df_co_cmd = df_co_cmd[~df_co_cmd.UIC.isin(df_fsc_dir_av.UIC.unique())].copy()
    # Create a DataFrame by filtering for BN level HQ ompanies and call it df_hhc_bn
    df_hhc_bn = df_co_cmd[df_co_cmd.UIC.str.endswith('T0')].copy()
    # Assign the NODETYPE: HHC_BN
    df_hhc_bn['NODETYPE'] = 'HHC_BN'
    # Filter the df_hhc_bn UICs out of df_co_cmd
    df_co_cmd = df_co_cmd[~df_co_cmd.UIC.isin(df_hhc_bn.UIC.unique())].copy()
    # Create a DataFrame by filtering for platoon or detachment level units and call it df_plt_det
    # Be sure not to overwrite any of the UICs in df_det_Bde
    df_plt_det = dfh_old[(dfh_old.SIMPLENAME.str.contains('Plt|Det')) & (~dfh_old.UIC.isin(df_det_bde.UIC.unique()))].copy()
    # Assign the NODETYPE: PLT_DET
    df_plt_det['NODETYPE'] = 'PLT_DET'
    # Filter the df_plt_det UICs out of df_co_cmd
    df_co_cmd = df_co_cmd[~df_co_cmd.UIC.isin(df_plt_det.UIC.unique())]
    # Create a DataFrame by filtering for augmentation companies to BDE and call it df_co_aug
    df_co_aug = df_co_cmd[df_co_cmd.DOCTPE =='AUGTDA'].copy()
    # Assign the NODETYPE: CO_AUG
    df_co_aug['NODETYPE'] = 'CO_AUG'
    # Filter the df_co_aug UICs out of df_co_cmd
    df_co_cmd = df_co_cmd[~df_co_cmd.UIC.isin(df_co_aug.UIC.unique())].copy()
    
    # Collect all DataFrames into a list for concatenation
    df_list =[df_bde_cmd, df_bn_cmd, df_co_cmd, df_co_aug, df_plt_det,
              df_hhc_bde, df_co_bde_dir_av, df_det_bde, df_hqstb, df_fsc_dir_av,
              df_hhc_bn]
       
    # Concatenate to dfc
    actionccsmall= f"Concatenating all DataFrames for Fiscal Year 20{fy}..."
    ccsmall = time_start(actionccsmall,nest=nest*4)
    dfc = pd.concat(df_list)

    time_stop(ccsmall,action=actionccsmall,nest=nest*4)
    time_stop(bigl,action=action2,nest=nest*3)

    ### Build hier_node_dict
    co_cmd_uics = dfc[dfc.NODETYPE == 'CO_CMD']['UIC'].unique().tolist()
    bn_cmd_uics = dfc[dfc.NODETYPE == 'BN_CMD']['UIC'].unique().tolist()
    bde_cmd_uics = dfc[dfc.NODETYPE == 'BDE_CMD']['UIC'].unique().tolist()
    hier_node_dict =  {co_uic:{'co_cmd':co_uic,'bn_cmd':np.nan,'bn_name':np.nan,'bde_cmd':np.nan,'bde_name':np.nan,'div_cmd':np.nan,'div_name':np.nan} for co_uic in co_cmd_uics}
    hier_node_dict.update({bn_uic:{'co_cmd':np.nan,'bn_cmd':bn_uic,'bn_name':np.nan,'bde_cmd':np.nan,'bde_name':np.nan,'div_cmd':np.nan,'div_name':np.nan} for bn_uic in bn_cmd_uics})
    hier_node_dict.update({bde_uic:{'co_cmd':np.nan,'bn_cmd':np.nan,'bn_name':np.nan,'bde_cmd':bde_uic,'bde_name':np.nan,'div_cmd':np.nan,'div_name':np.nan} for bde_uic in bde_cmd_uics})
    hier_node_dict.update({div_uic:{'co_cmd':np.nan,'bn_cmd':np.nan,'bn_name':np.nan,'bde_cmd':np.nan,'bde_name':np.nan,'div_cmd':div_uic,'div_name':np.nan} for div_uic in div_uics})
    
    df_hnd = pd.DataFrame.from_dict(hier_node_dict,orient='index').reset_index()
    df_hnd.columns = ['UIC']+ df_hnd.columns.tolist()[1:]
    
    # Create Battalion, Brigade, and Division name DataFrames for merging
    df_bn_name = dfc[dfc.NODETYPE=='BN_CMD'][['UIC','SIMPLENAME']].rename(columns = {'SIMPLENAME':'bn_name'})
    df_bde_name = dfc[dfc.NODETYPE=='BDE_CMD'][['UIC','SIMPLENAME']].rename(columns = {'SIMPLENAME':'bde_name'})
    df_div_name = df_div_tups[['uic','div_name']].rename(columns = {'uic':'UIC'})
    
    
    # Create a df_parent as a parent lookup datframe from fdc
    df_parent = dfc[['UIC','PARENTUIC']].copy()
    
    # Create a temporary DataFrame to hold the results of the lookups
    df_temp = df_hnd.copy()
    
    # --- 1. Fill 'bn_cmd' for the companies ---
    df_temp['bn_cmd'] = df_temp['bn_cmd'].fillna(
        df_hnd['co_cmd'].map(df_parent.set_index('UIC')['PARENTUIC']))
    # Create a lookup series from df_bn_name
    bn_lookup_series = df_bn_name.set_index('UIC')['bn_name']
    # Fill the NaN values in 'bn_name' using map()
    df_temp['bn_name'] = df_temp['bn_name'].fillna(
        df_temp['bn_cmd'].map(bn_lookup_series)) 
    
    # --- 2. Fill 'bde_cmd' for the companies and battalions ---
    df_temp['bde_cmd'] = df_temp['bde_cmd'].fillna(
        df_temp['bn_cmd'].map(df_parent.set_index('UIC')['PARENTUIC']))
    # Create a lookup series from df_bde_name
    bde_lookup_series = df_bde_name.set_index('UIC')['bde_name']
    # Fill the NaN values in 'bde_name' using map()
    df_temp['bde_name'] = df_temp['bde_name'].fillna(
        df_temp['bde_cmd'].map(bde_lookup_series)) 
    
    # --- 3. Fill 'div_uic' for the companies, battalions, and brigades ---
    df_temp['div_cmd'] = df_temp['div_cmd'].fillna(
        df_temp['bde_cmd'].map(df_parent.set_index('UIC')['PARENTUIC']))
    # Create a lookup series from df_div_name
    div_lookup_series = df_div_name.set_index('UIC')['div_name']
    # Fill the NaN values in 'bde_name' using map()
    df_temp['div_name'] = df_temp['div_name'].fillna(
        df_temp['div_cmd'].map(div_lookup_series)) 
    
    # --- 4. Add the 'fy' column ---
    df_temp['fy'] = int_fy
   
    # --- 5. Add the UTSIZE column ---
    df_temp = df_temp.merge(dfh_new,how='left',on='UIC')
    
    # --- 6. Add the fiscal year nodes DataFrame to the list for later concatenation ---
    uic_hierarchy_fy_list.append(df_temp)
    
    # --- 7. Save the fiscal year nodes DataFrame
    store_feather(df_temp,f"df_uic_hierarchy_nodes_20{fy}",uic_dir)
    
if fy_rng:
    # Concatenate all dataframes
    actionCC = f"Concatenating DataFrames for ALL Fiscal Years (20{fy_start}-20{fy_end})"
    cc = time_start(actionCC,nest=nest*3)
    df_master_concat = pd.concat(uic_hierarchy_fy_list)
    time_stop(cc,action=actionCC,nest=nest*3)
    store_feather(df_master_concat,f"df_uic_hierarchy_nodes_20{fy_start}-20{fy_end}",uic_dir)
time_stop(bigbigl,action=actionG,nest=nest*2)

In [ ]:
# Load and store all of the newer tables as fy dataframes and as a concatenated whole
WDARFF_df_list = []
# Read in the fiscal year data from the newer tables
table_files_dir = './data_imported/hierarchy_data/HUICs'
dir_table_files =[table for table in os.listdir(table_files_dir) if table.endswith('.txt')]
fy_list = []
df_new = None
for table_file in dir_table_files:
    fy = int((table_file.split('.')[0][-2:]))
    fy_list.append(fy)
    dfh_new = pd.read_csv(os.path.join(table_files_dir,table_file),sep='\t')
    fy = str(dfh_new.FYDATE.unique().tolist()[0])[2:4]
    dfh_new['fy'] = fy
    # Store the table as a DataFrame and add to list
    store_feather(dfh_new,f'df_WDARFF_20{fy}', uic_dir)
    WDARFF_df_list.append(dfh_new)
df_WDARFF_concat = pd.concat(WDARFF_df_list)
store_feather(df_WDARFF_concat,f"df_WDARFF_20{min(fy_list)}-20{max(fy_list)}",uic_dir)

# time_stop(bigbigbigl,action=actionB,nest=nest*1)

In [ ]:
df_WDARFF_concat.columns

### Node work with df_master_concat

In [ ]:
dfmc = df_master_concat.copy()
dfmc

### Work to utilize the new BIG UIC Tables

In [ ]:
dfx = load_feather('df_uic_hierarchy_nodes_2018-2026',uic_dir)
# dfx['echelon'] = np.nan
# for fy in range(2018,2027):
#     display(fy,dfx[(dfx.co_cmd.notna()) & (dfx.bn_cmd.isna()) & (dfx.fy == fy)])

dfx[(dfx.co_cmd.notna()) & (dfx.bn_cmd.isna())]
dfco = dfx.dropna(subset = 'co_cmd')
dfco[dfco.bn_cmd.isna()]
dfbn = dfx[(dfx.co_cmd.isna()) & (dfx.bn_cmd.notna())]
for fy in range(2018,2027):
    display(f" For FY: {fy} , No. of unique Bn UICs: {dfbn[dfbn.fy == fy]['UIC'].nunique()}")

In [ ]:
dfx

In [ ]:

fy = 15
uic_data_dir2 = './data_imported/hierarchy_data/HUICs'
file_name2 = f'xxxxx_20{fy}.txt'
dfh_new = pd.read_csv(os.path.join(uic_data_dir2,file_name2),sep='\t')
# dfc = load_feather('df_uic_hierarchy_2026',uic_dir)
# dfh_new = dfh_new[dfh_new.SERVICE == 'ARMY'].reset_index(drop=True)[['UIC','UTSIZE']].copy()
# dfc = dfc.merge(dfh_new,on='UIC',how='left')
# dfc
dfh_new[dfh_new.UTSIZE == 'SQDRN'].sort_values(by='OFF')

### Concatenated DataFrame further cleanup code

In [ ]:
# # Now look for anything in the CMD nodetypes that has an 'HH' in it and put it into df_hh_scraps
# cmd_nodes = [ 'BN_CMD', 'CO_CMD']
# dfc_cmd = dfc[dfc.NODETYPE.isin(cmd_nodes)].copy()
# dfc_cmd[dfc_cmd.DRRSANAME.str.contains('HH',na=False)]
# """  WDHEFF is the sustainment bde for 4id.  IT should be a BDE_CMD, right?
# WAMQAA  is the HQ and STB for the 1ID sustainment BDE  it should be an HQSTB I think"""

In [ ]:
# dfc[dfc.NODETYPE == 'HQSTB'].SIMPLENAME.value_counts().to_dict()

In [ ]:
store_feather(dfc,'testdf',uic_dir)

In [ ]:
store_feather(df_div_tups,'df_div_tups')

In [ ]:
table_files_dir = './data_imported/hierarchy_data/HUICs'
dir_table_files =[table for table in os.listdir(table_files_dir) if table.endswith('.txt')]
fy_list = [];table_list = []
df_new = None
for table_file in dir_table_files:
    fy = int((table_file.split('.')[0][-2:]))
    fy_list.append(fy)
    dfh_new = pd.read_csv(os.path.join(table_files_dir,table_file),sep='\t')
    table_list.append(dfh_new)
dfc = pd.concat(table_list)
print(fy_list)
dfc = dfc[(dfc.SERVICE == 'ARMY') &( dfc.GFMCOMPO == 'REGUL')].drop(columns = ['ID','PARENTID','EDATE']).sort_values(by='FYDATE')

bad_strings_SD = 'EN Div|ENDiv'
dfc = dfc[~dfc.SHORTDSCR.str.contains(bad_strings_SD)]
dfd = dfc[dfc.UTSIZE == 'DIV'].sort_values(by=['FYDATE','UIC'])
dfd.head(50)

In [ ]:
dfw = load_feather('2_UIC_hier_data/df_WDARFF_2015-2026')
dfw = dfw[(dfw.GFMCOMPO =='REGUL')].drop_duplicates()
dfw = move_column_after(dfw,'fy','RELDPTH')
dfw.fy.unique()

In [ ]:
dfx = (dfw[(dfw.UTSIZE == 'BDE') & (~dfw.LONGDSCR.str.contains('Engineer'))]
    .drop(columns=['ID','PARENTID','OFF','WO','ENL','CIV','AUTHTOT','EDATE','CATCODE',
                   'SERVICE','UTGFMSIZE','FYDATE','ORGTYPENAME','GFMCOMPO','RELDPTH'])
    .sort_values(by=['fy','SHORTDSCR'])
      )
dfx['fy'] = dfx['fy'].apply(lambda x: '20'+x )
dfx[(dfx.SHORTDSCR.str.contains('10 M')) & (~dfx.SHORTDSCR.str.contains('HHC'))]
# dfxx = dfx.drop(columns=[''])

In [ ]:
dfmc['fy'] = dfmc['fy'].astype(str)

In [ ]:
dfj = dfx.merge(dfmc,how='inner',on=['UIC','fy']).sort_values(by=['fy','UIC'])
dfj.head(20)

In [ ]:
# df_101.columns.tolist()

In [ ]:
df_101 = load_feather("df_with_met_101ABN_perc_cpts_SNAP_2000-12-31_to_2022-06-30_OER_2010-01-01_to_2019-01-01")
import matplotlib.pyplot as plt
plt.hist(df_101.cum_tb_rcvd_ratio_snr,bins=25)
df_101.cum_tb_rcvd_ratio_snr.value_counts()

In [ ]:
s1 = {1,2,3,4,5,6,7,8,9};s2={5,6,7,8,9,10,11,12}
s1 & s2